# 02 — Chunking Experiments

## Document → Chunks

**Experiment IDs:** CHK-001, CHK-002, CHK-003

### Objective

Evaluate how different `chunk_size` and `chunk_overlap` configurations affect the number of chunks generated from the Pharma Sales dataset.

### Why Chunking?

Notebook 01 established that the dataset contains approximately 2,085-character documents.

Before generating embeddings, we need to determine whether splitting these documents into smaller pieces provides a useful unit for semantic retrieval.

### Questions We Want to Answer

1. How many chunks are produced by each configuration?
2. How does `chunk_size` affect document fragmentation?
3. How does `chunk_overlap` affect the resulting chunks?
4. Which configuration should become our initial baseline?

### Experiment Principle

We change only the chunking configuration while keeping the source dataset constant.

This gives us a controlled comparison between configurations.

## 1. Environment & Imports

We reuse the same project environment established in Notebook 01.

`RecursiveCharacterTextSplitter` is used to split documents while attempting to preserve natural text boundaries.

In [10]:
# Standard library
import os

# Environment configuration
from dotenv import load_dotenv

# Data analysis
import pandas as pd

# LangChain document loader and text splitter
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Load the project environment.
load_dotenv()

# Validate the environment used by downstream notebooks.
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY environment variable is not set. "
        "Please configure it in your .env file."
    )

print("Environment configured successfully.")

Environment configured successfully.


## 2. Load the Canonical Dataset

The same canonical dataset from Notebook 01 is used here.

Keeping the source dataset unchanged ensures that the experimental variable is the chunking configuration rather than the input data.

In [11]:
DATA_PATH = "../data/Pharma_Sales_Long.csv"

loader = CSVLoader(
    file_path=DATA_PATH,
    encoding="utf-8"
)

data = loader.load()

print(f"Loaded {len(data)} documents.")

Loaded 300 documents.


## 3. Validate the Source Dataset

The chunking experiment should operate on the same 300 documents established in Notebook 01.

If the document count changes, the results are no longer directly comparable with the ingestion baseline.

In [12]:
assert len(data) == 300, (
    f"Expected 300 documents, but found {len(data)}."
)

assert all(
    document.page_content.strip()
    for document in data
), "One or more documents contain empty page_content."

print("✓ Source dataset validation passed.")
print(f"✓ Documents available for chunking: {len(data)}")

✓ Source dataset validation passed.
✓ Documents available for chunking: 300


## 4. Establish the Source Baseline

Notebook 01 established an average document length of approximately 2,085 characters.

We calculate it again here so this notebook remains independently understandable.

In [13]:
doc_lengths = [
    len(document.page_content)
    for document in data
]

average_document_length = sum(doc_lengths) / len(doc_lengths)

print(f"Average document length: {average_document_length:.2f} characters")

Average document length: 2084.79 characters


## 5. Define the Experimental Configurations

We evaluate three configurations.

| Experiment | Chunk Size | Chunk Overlap |
|---|---:|---:|
| CHK-001 | 250 | 20 |
| CHK-002 | 500 | 50 |
| CHK-003 | 1,000 | 100 |

These provide small, medium, and large chunking strategies relative to the approximately 2,085-character source documents.

The overlap increases with chunk size so adjacent chunks retain some shared context.

In [28]:
chunking_configs = [
    {
        "experiment_id": "CHK-001",
        "chunk_size": 250,
        "chunk_overlap": 20,
    },
    {
        "experiment_id": "CHK-002",
        "chunk_size": 500,
        "chunk_overlap": 50,
    },
    {
        "experiment_id": "CHK-003",
        "chunk_size": 1000,
        "chunk_overlap": 100,
    },
]

for config in chunking_configs:
    print(
        f"{config['experiment_id']} | "
        f"chunk_size={config['chunk_size']} | "
        f"chunk_overlap={config['chunk_overlap']}"
    )

CHK-001 | chunk_size=250 | chunk_overlap=20
CHK-002 | chunk_size=500 | chunk_overlap=50
CHK-003 | chunk_size=1000 | chunk_overlap=100


## 6. Run the Chunking Experiments

For each configuration we:

1. Create a new `RecursiveCharacterTextSplitter`.
2. Split the same source documents.
3. Record the resulting chunk count.
4. Preserve the generated chunks for inspection.

No embeddings or retrieval are performed in this notebook.

In [29]:
experiment_results = []
chunk_sets = {}

for config in chunking_configs:
    # Create a fresh splitter so each experiment remains isolated.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"]
    )

    chunks = text_splitter.split_documents(data)

    chunk_sets[config["experiment_id"]] = chunks

    experiment_results.append({
        "Experiment_ID": config["experiment_id"],
        "Chunk_Size": config["chunk_size"],
        "Chunk_Overlap": config["chunk_overlap"],
        "Documents": len(data),
        "Chunks": len(chunks),
    })

    print(
        f"{config['experiment_id']} | "
        f"chunk_size={config['chunk_size']} | "
        f"overlap={config['chunk_overlap']} | "
        f"chunks={len(chunks)}"
    )

CHK-001 | chunk_size=250 | overlap=20 | chunks=2983
CHK-002 | chunk_size=500 | overlap=50 | chunks=1538
CHK-003 | chunk_size=1000 | overlap=100 | chunks=900


## 7. Compare Chunking Results

The first evaluation metric is the number of generated chunks.

A larger number of chunks means greater document fragmentation. A smaller number means larger pieces of text are retained together.

In [16]:
results_df = pd.DataFrame(experiment_results)

display(results_df)

,Experiment_ID,Chunk_Size,Chunk_Overlap,Documents,Chunks
0,CHK-001,250,20,300,2983
1,CHK-002,500,50,300,1538
2,CHK-003,1000,100,300,900


## 8. Inspect Sample Chunks

Chunk counts alone do not tell us whether the splitting behavior is sensible.

We therefore inspect a few chunks from `CHK-002`, our initial baseline candidate.

In [17]:
baseline_chunks = chunk_sets["CHK-002"]

for index, chunk in enumerate(baseline_chunks[:3], start=1):
    print("=" * 80)
    print(f"Chunk {index}")
    print(f"Length   : {len(chunk.page_content)} characters")
    print(f"Metadata : {chunk.metadata}")
    print("Content:")
    print(chunk.page_content)

Chunk 1
Length   : 257 characters
Metadata : {'source': '../data/Pharma_Sales_Long.csv', 'row': 0}
Content:
Transaction_ID: TXN00001
Product: GARDASIL 9
Business_Unit: Vaccines
Indication: HPV Prevention
Region: West
State: Maharashtra
Territory: T001
HCP_Speciality: Surgeon
Sales_Representative: Rep_1
Quantity: 12
Sales_Value: 661097
Transaction_Date: 2026-01-17
Chunk 2
Length   : 493 characters
Metadata : {'source': '../data/Pharma_Sales_Long.csv', 'row': 0}
Content:
Notes: Product Overview: GARDASIL 9 is used in HPV Prevention. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,
Chunk 3
Length   : 497 character

## 9. Chunk Length Analysis

We inspect the resulting chunk lengths for each configuration.

This provides additional context beyond the total number of chunks.

In [18]:
for experiment_id, chunks in chunk_sets.items():
    lengths = [len(chunk.page_content) for chunk in chunks]

    print("=" * 70)
    print(experiment_id)
    print(f"Minimum chunk length : {min(lengths)}")
    print(f"Maximum chunk length : {max(lengths)}")
    print(f"Average chunk length : {sum(lengths) / len(lengths):.2f}")

CHK-001
Minimum chunk length : 35
Maximum chunk length : 250
Average chunk length : 221.88
CHK-002
Minimum chunk length : 62
Maximum chunk length : 499
Average chunk length : 434.12
CHK-003
Minimum chunk length : 239
Maximum chunk length : 999
Average chunk length : 725.62


## 10. Findings

### Observations

1. `CHK-001` produces the highest number of chunks because it uses the smallest chunk size.
2. `CHK-003` produces the fewest chunks because it uses the largest chunk size.
3. `CHK-002` sits between the two extremes.
4. Chunk overlap preserves some shared context between neighboring chunks.
5. Higher overlap can also increase duplicated content across chunks.

### Important Learning

Chunk count is an operational metric, but it is **not sufficient to select the final configuration**.

The selected configuration must eventually be evaluated through embedding and retrieval experiments.

## 11. Decision

### Initial Baseline: CHK-002

For the next stage, we will use:

```text
Chunk Size    : 500
Chunk Overlap : 50
Documents     : 300
Chunks        : 1,538
```

### Rationale

`CHK-001` creates substantially more chunks and therefore greater fragmentation.

`CHK-003` creates fewer chunks but retains larger pieces of text.

`CHK-002` provides a practical middle ground and will therefore be used as the **initial baseline**, not as a final optimized configuration.

The embedding and retrieval experiments may change this decision if retrieval quality indicates that another configuration performs better.

## 12. Persist Experiment Results

The chunking results are written to the project's central `experiment_log.csv`.

In [21]:
## Persist Experiment Results

EXPERIMENT_LOG_PATH = "../experiments/experiment_log.csv"

chunk_log = results_df.copy()

chunk_log["Area"] = "Chunking"

chunk_log["Configuration"] = chunk_log.apply(
    lambda row: (
        f"chunk_size={row['Chunk_Size']}; "
        f"chunk_overlap={row['Chunk_Overlap']}"
    ),
    axis=1,
)

chunk_log["Question"] = ""
chunk_log["Top_K"] = ""
chunk_log["Results"] = chunk_log["Chunks"].astype(str)

chunk_log["Observation"] = chunk_log["Experiment_ID"].map({
    "CHK-001": "Smallest chunk size produced the highest number of chunks.",
    "CHK-002": "Middle configuration produced 1,538 chunks and is the initial baseline.",
    "CHK-003": "Largest chunk size produced the fewest number of chunks.",
})

chunk_log["Conclusion"] = chunk_log["Experiment_ID"].map({
    "CHK-001": (
        "Higher fragmentation; evaluate only if retrieval benefits "
        "from smaller chunks."
    ),
    "CHK-002": (
        "Use as the initial baseline for embedding and retrieval experiments."
    ),
    "CHK-003": (
        "Lower fragmentation; evaluate only if larger context "
        "improves retrieval."
    ),
})

experiment_log_columns = [
    "Experiment_ID",
    "Area",
    "Configuration",
    "Question",
    "Documents",
    "Chunks",
    "Top_K",
    "Results",
    "Observation",
    "Conclusion",
]

chunk_log = chunk_log[experiment_log_columns]


# If an experiment log already exists, preserve other experiments.
# If the file is empty or invalid, start with the current chunking experiments.
if os.path.exists(EXPERIMENT_LOG_PATH) and os.path.getsize(EXPERIMENT_LOG_PATH) > 0:

    existing_log = pd.read_csv(EXPERIMENT_LOG_PATH)

    # Remove previous CHK records so rerunning this notebook
    # does not create duplicate experiment entries.
    if "Experiment_ID" in existing_log.columns:
        existing_log = existing_log[
            ~existing_log["Experiment_ID"].isin(
                ["CHK-001", "CHK-002", "CHK-003"]
            )
        ]

        combined_log = pd.concat(
            [existing_log, chunk_log],
            ignore_index=True
        )
    else:
        combined_log = chunk_log

else:
    # Create the experiment log when the file is missing or empty.
    combined_log = chunk_log


combined_log.to_csv(
    EXPERIMENT_LOG_PATH,
    index=False
)

print(f"Experiment log updated: {EXPERIMENT_LOG_PATH}")
print(f"Total logged experiments: {len(combined_log)}")

Experiment log updated: ../experiments/experiment_log.csv
Total logged experiments: 3


## 13. Final Validation

The notebook finishes with assertions confirming that all three experiments executed successfully and produced the expected ordering of chunk counts.

In [ ]:
# Final experiment-level validation.
assert len(experiment_results) == 3

assert all(
    result["Documents"] == 300
    for result in experiment_results
)

assert all(
    result["Chunks"] > 0
    for result in experiment_results
)

# For this dataset, smaller chunks should produce more chunks.
chunk_counts = [
    result["Chunks"]
    for result in experiment_results
]

assert chunk_counts[0] > chunk_counts[1] > chunk_counts[2]

print("✓ Chunking experiment validation passed.")
print("✓ CHK-001, CHK-002 and CHK-003 completed.")
print("✓ Initial baseline: CHK-002 (500/50).")
print("✓ Ready for Notebook 03 — Embedding Experiments.")

## 14. Handoff to Notebook 03

The chunking stage is complete.

```text
300 Documents
      ↓
3 Chunking Configurations
      ↓
CHK-002 Selected as Initial Baseline
      ↓
500 Character Chunk Size
50 Character Overlap
1,538 Chunks
      ↓
Embedding Experiments
```

Notebook 03 will evaluate embedding model options using the selected chunking baseline.